# Run one run card (Kaggle, "Save & Run All")

Fill in the first cell, attach the datasets the card needs (**Add Input**), set the
accelerator the card names (**GPU T4 x2**, or none for a CPU card) and turn **Internet on**,
then **Save Version -> Save & Run All**. The run needs no one at the keyboard. At the end it
prints the card's section 6; the files to keep are in the notebook's output:

- `run_report_<card>.md` - paste it into section 6 of the card;
- `results_to_commit/` - the results files to commit to `research-loop` (docs/team_setup.md);
- `handoff/<name>/` - each folder becomes ONE **private** Kaggle dataset for the next account.

See `docs/team_setup.md` for the whole procedure. Nothing here is ever made public.

In [ ]:
# ---- fill in ----------------------------------------------------------------------------
CARD = "RC-E2"            # the run card, e.g. RC-D1, RC-E0b, RC-E1, RC-E2, RC-E5 ...
SEED = 0                  # 0, 1 or 2 for cards split by seed; None for the others
DATA = "/kaggle/input/medcompress-data"   # the shared-data dataset; None for RC-D1 itself
FROM = []                 # hand-off datasets, e.g. ["/kaggle/input/e0-teacher-cache"]
COMMIT = None             # a commit of research-loop to run; None = the latest
EXTRA = []                # e.g. ["--kvasir-seg", "/kaggle/input/kvasir-seg"] for RC-D1

In [ ]:
import os
os.chdir("/kaggle/working")
!git clone -q -b research-loop https://github.com/Soheil-jafari/medvlm-compression-audit.git
%cd /kaggle/working/medvlm-compression-audit
if COMMIT:
    !git checkout -q {COMMIT}
!git log -1 --format=%H

In [ ]:
# Kaggle ships torchao 0.10.0; peft 0.21.2 needs 0.16.0 or newer (RC-M1): remove the old one
from importlib import metadata
try:
    old = tuple(int(x) for x in metadata.version("torchao").split(".")[:2]) < (0, 16)
except metadata.PackageNotFoundError:
    old = False
if old:
    !pip uninstall -y -q torchao
!pip install -q -r requirements/base.txt -r requirements/train.txt && pip install -q -e . --no-deps

In [ ]:
import shlex, subprocess, sys
args = ["--card", CARD]
if SEED is not None:
    args += ["--seed", str(SEED)]
if DATA:
    args += ["--data-root", DATA]
for path in FROM:
    args += ["--from", path]
args += list(EXTRA)
print("python scripts/run_card.py " + " ".join(shlex.quote(a) for a in args))
code = subprocess.run([sys.executable, "scripts/run_card.py", *args]).returncode
print(f"run_card.py finished with exit code {code}" + ("" if code == 0 else ": CARD NOT COMPLETE, see the report"))